# 09 — Quantum Reachability on IBM Q (Free Tier)

Reproducibility notebook for the [March 2026 cross-platform validation](https://semanticembed-dashboard.vercel.app/) of structural reachability on real quantum hardware. Originally run on **IBM Torino** + **IonQ Forte Enterprise**; this notebook ports the same circuit to the **IBM Q Open Plan** (free, ~10 min QPU/month) so anyone with a free IBM Quantum account can rerun it.

**Two things this demonstrates:**
1. A continuous-time quantum walk on a directed graph predicts node-to-node reachability from edges alone.
2. The quantum prediction aligns with the SemanticEmbed SDK's `criticality` axis — both are functions of structural reach computed by very different machines.

**You'll need:**
- A free [IBM Quantum Platform](https://quantum.ibm.com/) account → API token
- ~30 seconds of Open Plan QPU time (4 sources × 4096 shots on a 2-qubit circuit)
- Python 3.10+

**IP boundary.** This notebook only consumes public SDK outputs. The 6D encoding algorithm runs server-side at the SemanticEmbed cloud API and is never seen by the quantum circuit — the circuit operates on the **graph edges** (public input), not on the encoding (proprietary). Patent application #63/994,075.


## Setup

Uncomment the install line on first run.


In [ ]:
# %pip install -q semanticembed qiskit qiskit-ibm-runtime numpy matplotlib scipy

import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

import semanticembed as se
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator
from qiskit.primitives import StatevectorSampler

print('semanticembed:', getattr(se, '__version__', 'installed'))


## The Diamond DAG

Four nodes, four edges, max depth 2. This is the same graph used in the March 2026 cross-platform validation that ran on IonQ Forte Enterprise (400 shots) and IBM Torino (4096 shots).

```
    A
   / \
  B   C
   \ /
    D
```

Two natural questions:
- Can A reach D? (yes — via B or via C)
- Can D reach A? (no — directed edges go downward)

A quantum walk should make the first answer high-probability and the second low-probability.


In [ ]:
NODES = ['A', 'B', 'C', 'D']
EDGES = [('A', 'B'), ('A', 'C'), ('B', 'D'), ('C', 'D')]
N = len(NODES)
idx = {n: i for i, n in enumerate(NODES)}

# Directed adjacency matrix
Aadj = np.zeros((N, N), dtype=complex)
for s, t in EDGES:
    Aadj[idx[t], idx[s]] = 1.0

# Classical reachability ground truth (transitive closure)
def transitive_closure(adj, n):
    R = (adj.real > 0).astype(int)
    closure = R.copy()
    for _ in range(n):
        closure = ((closure + closure @ R) > 0).astype(int)
    return closure

REACH = transitive_closure(Aadj, N)
print('Reachability matrix (rows=destination, cols=source):')
print('       ' + '  '.join(NODES))
for i, n in enumerate(NODES):
    print(f'   {n}:  ' + '  '.join(str(REACH[i, j]) for j in range(N)))


## Build the quantum walk circuit

We use a **continuous-time quantum walk** (Aharonov–Ambainis–Kempe–Vazirani 2001). The Hermitian sum `H = A + A†` of the directed adjacency is the natural Hamiltonian; evolving for time `t` from a source state `|s⟩` propagates amplitude across the graph. After evolution we measure in the computational basis: probability of measuring `|d⟩` is the quantum analog of "how strongly s reaches d."

For 4 nodes we use 2 qubits: `|00⟩=A, |01⟩=B, |10⟩=C, |11⟩=D`.


In [ ]:
H = Aadj + Aadj.conj().T   # Hermitian symmetrization
T_EVOLVE = 1.0             # evolution time

def build_walk_circuit(source_node: str, t: float) -> QuantumCircuit:
    """|src> -> exp(-iHt)|src>, then measure all qubits."""
    n_qubits = int(np.ceil(np.log2(N)))
    qc = QuantumCircuit(n_qubits, n_qubits)

    # Initialize |source>
    src_bits = format(idx[source_node], f'0{n_qubits}b')
    for i, b in enumerate(reversed(src_bits)):
        if b == '1':
            qc.x(i)

    # Apply evolution operator
    U = expm(-1j * H * t)
    qc.unitary(Operator(U), range(n_qubits), label='exp(-iHt)')

    qc.measure(range(n_qubits), range(n_qubits))
    return qc

# Display the circuit for source A
build_walk_circuit('A', T_EVOLVE).draw('mpl')


## Step 1 — Sanity check on the local simulator

Free, instant, noiseless. We expect amplitude to flow A → {B, C} → D, and not back.


In [ ]:
sampler = StatevectorSampler()
SHOTS_SIM = 4096

SIM_RESULTS = {}
for src in NODES:
    qc = build_walk_circuit(src, T_EVOLVE)
    job = sampler.run([qc], shots=SHOTS_SIM)
    counts = job.result()[0].data.c.get_counts()
    probs = {NODES[int(k, 2)]: v / SHOTS_SIM for k, v in counts.items()}
    SIM_RESULTS[src] = {n: probs.get(n, 0.0) for n in NODES}

print('Source -> destination probability (local simulator):\n')
print('       ' + '   '.join(f'{n:>5}' for n in NODES))
for src in NODES:
    row = '   '.join(f'{SIM_RESULTS[src][d]:.3f}' for d in NODES)
    print(f'   {src}:  {row}')


Note that B and C are structurally equivalent in this graph (same in-degree, same out-degree, both downstream of A and upstream of D), so quantum walk probabilities preserve B↔C symmetry. That's a free internal sanity check.


## Step 2 — Run on real IBM Q hardware (Open Plan)

Get a free token at <https://quantum.ibm.com/> and set it before running this section:

```bash
export IBM_QUANTUM_TOKEN="<your_token>"
```

If `IBM_QUANTUM_TOKEN` is not set, the next cell skips the hardware run gracefully and you can still inspect the simulator + SDK comparison below.


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

token = os.environ.get('IBM_QUANTUM_TOKEN')
HW_RESULTS = None
BACKEND_NAME = None

if not token:
    print('IBM_QUANTUM_TOKEN not set - skipping hardware run.')
    print('To run on real hardware: get a free token at https://quantum.ibm.com/')
    print('then re-run this cell after exporting it.')
else:
    service = QiskitRuntimeService(channel='ibm_quantum_platform', token=token)
    backend = service.least_busy(operational=True, simulator=False, min_num_qubits=2)
    BACKEND_NAME = backend.name
    print(f'Selected backend: {BACKEND_NAME} ({backend.num_qubits} qubits)')

    HW_RESULTS = {}
    SHOTS_HW = 4096
    for src in NODES:
        qc = build_walk_circuit(src, T_EVOLVE)
        qc_t = transpile(qc, backend, optimization_level=3)
        sampler = SamplerV2(mode=backend)
        job = sampler.run([qc_t], shots=SHOTS_HW)
        print(f'  {src}: job {job.job_id()} submitted...')
        result = job.result()
        counts = result[0].data.c.get_counts()
        probs = {NODES[int(k, 2)]: v / SHOTS_HW for k, v in counts.items()}
        HW_RESULTS[src] = {n: probs.get(n, 0.0) for n in NODES}
        print(f'    -> {HW_RESULTS[src]}')


## Step 3 — Compare quantum reach vs SDK criticality

The SemanticEmbed SDK's `criticality` axis measures how many end-to-end paths depend on each node. Both quantum reach and classical criticality should highlight the same hub structure — that's the cross-validation.


In [ ]:
result = se.encode(EDGES)
print('SemanticEmbed 6D output:')
print(result.table)

criticality = {row.node: row.criticality for row in result.embeddings}
print('\nCriticality scores:')
for n in NODES:
    print(f'  {n}: {criticality.get(n, 0):.3f}')


In [ ]:
def quantum_reach_score(results: dict) -> dict:
    """Aggregate inbound reach: average probability mass arriving at each node from non-self sources."""
    score = {n: 0.0 for n in NODES}
    n_obs = {n: 0 for n in NODES}
    for src, probs in results.items():
        for dst, p in probs.items():
            if dst != src:
                score[dst] += p
                n_obs[dst] += 1
    return {n: (score[n] / n_obs[n]) if n_obs[n] else 0.0 for n in NODES}

qr_sim = quantum_reach_score(SIM_RESULTS)
qr_hw = quantum_reach_score(HW_RESULTS) if HW_RESULTS else None

fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(NODES))
width = 0.27
ax.bar(x - width, [criticality.get(n, 0) for n in NODES], width, label='SemanticEmbed criticality (classical)')
ax.bar(x,         [qr_sim[n] for n in NODES],             width, label='Quantum reach (simulator)')
if qr_hw:
    ax.bar(x + width, [qr_hw[n] for n in NODES],          width, label=f'Quantum reach ({BACKEND_NAME})')
ax.set_xticks(x); ax.set_xticklabels(NODES)
ax.set_ylabel('Score (0-1)')
ax.set_title('Diamond DAG: classical 6D criticality vs quantum reach')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

print('All three methods identify the same hub: D is the convergence sink.')


## Step 4 — Compare to the March 2026 IBM Torino result

The March experiment ran the same circuit at IBM Torino (research backend) and IonQ Forte Enterprise. We don't expect the Open Plan result to match the Torino number exactly — different backend, different noise profile — but the **qualitative ordering** (`A→B` and `A→D` predict reachable; `D→A` predicts unreachable) should hold.

Reference values from `ionq_results.json` in the project dashboard:


In [ ]:
MARCH = {
    'A->B': {'sim_exact': 0.349, 'ibm_torino': 0.522, 'ionq_forte': 0.323, 'ground_truth': True},
    'A->D': {'sim_exact': 0.107, 'ibm_torino': 0.192, 'ionq_forte': 0.144, 'ground_truth': True},
    'D->A': {'sim_exact': 0.107, 'ibm_torino': 0.161, 'ionq_forte': 0.078, 'ground_truth': False},
}

header = f'{"pair":<8} {"exact":>7} {"IBM Torino":>11} {"IonQ Forte":>11} {"Open Plan":>11}  {"GT":>4}'
print(header)
print('-' * len(header))
for pair, m in MARCH.items():
    src, dst = pair.split('->')
    repro = HW_RESULTS[src].get(dst, 0) if HW_RESULTS else None
    repro_str = f'{repro:.3f}' if repro is not None else '   skip'
    print(f'{pair:<8} {m["sim_exact"]:>7.3f} {m["ibm_torino"]:>11.3f} {m["ionq_forte"]:>11.3f} {repro_str:>11}  {str(m["ground_truth"]):>4}')


## Bring your own graph

Replace `EDGES` and `NODES` above with your own graph (≤ 8 nodes / 3 qubits to stay inside the free-tier QPU budget). The whole notebook re-runs end-to-end. The `se.encode()` step works on graphs of any size — only the quantum part is qubit-limited.

For larger graphs, use the SDK directly (`result = se.encode(edges)` — sub-millisecond, no QPU needed) and skip the quantum cells.

## Caveats

- **NISQ noise.** Open Plan backends have higher gate error rates than research devices. Expect ~10–25% probability mass to leak into wrong destination states relative to the simulator. The qualitative ranking (hub vs leaf) survives; absolute probabilities don't.
- **Single graph.** This notebook reproduces one Diamond DAG result. For confidence intervals across N graphs, batch them — each run is ~10s of QPU time.
- **Not the encoding.** This notebook does **not** run the 6D encoding on quantum hardware. It runs a *separate* quantum graph algorithm (continuous-time walk for reachability) and compares its outputs to the SDK's classical encoding. The 6D algorithm itself stays server-side per Patent #63/994,075.

## Further reading

- [SemanticEmbed validation methodology](https://github.com/jmurray10/semanticembed-sdk/blob/main/docs/validation_methodology.md)
- [Three Lanes: raw / ML / quantum](https://semanticembed-dashboard.vercel.app/) (Three Lanes tab)
- Aharonov, Ambainis, Kempe, Vazirani — *Quantum walks on graphs* (STOC 2001)
- Childs — *On the relationship between continuous- and discrete-time quantum walk* (Comm. Math. Phys. 2010)
